# 🌙 03. Multi-Phase Solar Illumination & Grazing Incidence Analysis

**Mission Context**: Photometric correction and classification of Chandrayaan-2 imagery under varying solar incidence angles ($i = 15^\circ - 85^\circ$).  
**Objectives**:
- Compute mean luminance, brightness variance, RMS contrast, and shadow coverage.
- Classify images into **Easy**, **Moderate**, and **Difficult** registration tiers.
- Generate Photometric Heatmaps and histogram entropy plots.
- Export `illumination_report.csv` and `illumination_scores.json`.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import json

sys.path.append(str(Path.cwd().parent))
from lunar_core.config import load_config
from lunar_core.data_loader import LunarImageLoader
from lunar_core.illumination import IlluminationAnalyzer

config = load_config()
illum_analyzer = IlluminationAnalyzer(shadow_cutoff=35, highlight_cutoff=220)


In [ ]:
# Process dataset frames across different folders
img_paths = list(Path("data").glob("*/*.png"))[:15]
records = []

for p in img_paths:
    img = LunarImageLoader.load_image(str(p))
    if img is not None:
        metrics = illum_analyzer.compute_illumination_metrics(img)
        metrics["file_path"] = str(p)
        metrics["file_name"] = p.name
        metrics.pop("illumination_heatmap")
        records.append(metrics)

df_illum = pd.DataFrame(records)
df_illum.head()


In [ ]:
# Visualize Illumination Classes & Heatmaps
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Classification distribution
counts = df_illum['classification'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#10B981', '#F59E0B', '#EF4444'])
axes[0].set_title("Image Registration Difficulty Tiers", fontweight='bold')
axes[0].set_ylabel("Number of Frames")

# Shadow vs Contrast Scatter
axes[1].scatter(df_illum['shadow_coverage_pct'], df_illum['rms_contrast'], c=df_illum['difficulty_score'], cmap='coolwarm', s=80)
axes[1].set_title("Shadow % vs. RMS Contrast", fontweight='bold')
axes[1].set_xlabel("Shadow Coverage (%)")
axes[1].set_ylabel("RMS Contrast")

# Sample Heatmap
sample_img = LunarImageLoader.load_image(str(img_paths[0]))
hmap = illum_analyzer.compute_illumination_metrics(sample_img)["illumination_heatmap"]
im_h = axes[2].imshow(hmap, cmap='inferno')
axes[2].set_title("Solar Incidence Photometric Gradient", fontweight='bold')
plt.colorbar(im_h, ax=axes[2], fraction=0.046)
axes[2].axis('off')

plt.tight_layout()
os.makedirs("outputs/visualizations", exist_ok=True)
plt.savefig("outputs/visualizations/03_illumination_analysis.png", dpi=300)
plt.show()


In [ ]:
# Export reports
os.makedirs("outputs/reports", exist_ok=True)
df_illum.to_csv("outputs/reports/illumination_report.csv", index=False)

summary_scores = {
    "total_classified_images": len(df_illum),
    "tier_distribution": df_illum['classification'].value_counts().to_dict(),
    "average_shadow_pct": round(float(df_illum['shadow_coverage_pct'].mean()), 2),
    "average_contrast": round(float(df_illum['rms_contrast'].mean()), 2)
}
with open("outputs/reports/illumination_scores.json", "w") as f:
    json.dump(summary_scores, f, indent=4)

print("Exported outputs/reports/illumination_report.csv and illumination_scores.json")
